# Build a Haystack pipeline from local `src/` components

This notebook builds and runs a tiny Haystack pipeline with regular Python. It imports the repository's custom components directly from `src/` and does not load Hydra or any YAML configuration.

In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if candidate.joinpath("pyproject.toml").is_file() and candidate.joinpath("src").is_dir():
            return candidate
    raise RuntimeError("Could not find the repository root from this notebook.")


project_root = find_project_root(Path.cwd().resolve())
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

project_root

In [ ]:
from haystack import Document, Pipeline

from retrieval_components.components.indexing import JsonlDocumentIndexer
from retrieval_components.components.interfaces import InferenceInput, InferenceOutput
from retrieval_components.components.retrieval import JsonlKeywordRetriever

Create a tiny JSONL document index using the repository's JSONL indexer component. This is deliberately small and local so the example runs without downloading models.

In [ ]:
index_path = project_root / "artifacts" / "notebook_examples" / "keyword_index.jsonl"

documents = [
    Document(id="hydra", content="Hydra composes experiment configuration from reusable groups."),
    Document(id="haystack", content="Haystack pipelines connect retrieval components together."),
    Document(id="metrics", content="Retrieval experiments compare ranked documents with qrels."),
]

index_result = JsonlDocumentIndexer(output_path=str(index_path), overwrite=True).run(documents)
index_result

Build a Haystack pipeline directly in Python. The input/output boundary components mirror the stage pipeline shape, while the retriever reads from the JSONL artifact we just wrote.

In [ ]:
pipeline = Pipeline()
pipeline.add_component("input", InferenceInput())
pipeline.add_component("retriever", JsonlKeywordRetriever(index_path=str(index_path), top_k=2))
pipeline.add_component("output", InferenceOutput())

pipeline.connect("input.query", "retriever.query")
pipeline.connect("input.candidate_document_ids", "retriever.candidate_document_ids")
pipeline.connect("retriever.documents", "output.documents")

pipeline

Run a query. Because this stage-style graph connects `candidate_document_ids`, pass the candidate ids explicitly. In a full inference stage, those ids come from the input mapping.

In [ ]:
result = pipeline.run(
    {
        "input": {
            "query": "haystack retrieval pipeline",
            "candidate_document_ids": [document.id for document in documents],
        }
    }
)

[
    {"id": document.id, "score": document.score, "content": document.content}
    for document in result["output"]["documents"]
]